# Stoneforge RL — Kaggle Training Notebook

## Vorbereitung (einmalig)
1. Repo als ZIP als Kaggle Dataset hochladen (Name: **`stoneforge-rl`**)
2. Dieses Notebook mit dem Dataset verknüpfen: *Add Data → Your Datasets → stoneforge-rl*
3. GPU aktivieren: *Settings → Accelerator → GPU T4 x2*

## Was passiert automatisch:
- **Schritt 1** (~5s): Repo entpacken
- **Schritt 2** (~60s): System-Pakete installieren (cmake, g++)
- **Schritt 3** (~90s): Python-Pakete installieren
- **Schritt 4** (~120s): C++-Extension kompilieren (stoneforge_sim.so)
- **Schritt 5** (~10s): Build prüfen, GPU-Info
- **Schritt 6** (Stunden): Training mit Live-Progress + Log-Datei
- **Schritt 7** (~60s): Finale Evaluation + Zusammenfassung

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# KONFIGURATION — hier anpassen
# ═══════════════════════════════════════════════════════════════════════════════
ALGO        = "rppo"        # rppo | ppo | dqn | a2c
TIMESTEPS   = 3_000_000     # Kaggle hat ~12h → großzügig
N_ENVS      = 16            # mehr als lokal (Kaggle hat mehr CPU-Kerne)
EXIT_MIN    = 5             # Curriculum-Start
EXIT_MAX    = 12            # Curriculum-Start
EVAL_FREQ   = 25_000        # alle X Steps evaluieren
LOG_EVERY   = 10_000        # alle X Steps in Log schreiben
SAVE_DIR    = "/kaggle/working/models/rppo_kaggle"
LOG_FILE    = "/kaggle/working/training.log"
# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ── Hilfsfunktionen (Logging + Timing) ────────────────────────────────────────
import time, os, sys, logging
from datetime import datetime, timedelta

os.makedirs(SAVE_DIR, exist_ok=True)

# Logger der gleichzeitig in Datei und Notebook schreibt
logger = logging.getLogger("stoneforge")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)

fmt = logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s",
                        datefmt="%H:%M:%S")
fh.setFormatter(fmt)
ch.setFormatter(fmt)
logger.addHandler(fh)
logger.addHandler(ch)

def log(msg, level="info"):
    getattr(logger, level)(msg)

class Timer:
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t = time.time()
        log(f"▶  {self.label} ...")
        return self
    def __exit__(self, *_):
        self.elapsed = time.time() - self.t
        log(f"✓  {self.label} — {self.elapsed:.1f}s")

def fmt_time(seconds):
    return str(timedelta(seconds=int(seconds)))

log(f"Notebook gestartet — {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}")
log(f"Konfiguration: algo={ALGO}, timesteps={TIMESTEPS:,}, n_envs={N_ENVS}, "
    f"exit={EXIT_MIN}-{EXIT_MAX}")
log(f"Log-Datei: {LOG_FILE}")

In [ ]:
# ── Schritt 1: Repo entpacken ──────────────────────────────────────────────────
import zipfile, glob, shutil

DATASET_PATH = "/kaggle/input/stoneforge-rl"
WORK_DIR     = "/kaggle/working/stoneforge"

with Timer("Repo entpacken"):
    zips = glob.glob(f"{DATASET_PATH}/*.zip")
    if zips:
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK_DIR)
        log(f"  ZIP: {zips[0]} → {WORK_DIR}")
    else:
        shutil.copytree(DATASET_PATH, WORK_DIR, dirs_exist_ok=True)
        log(f"  Kopiert: {DATASET_PATH} → {WORK_DIR}")

    # Falls ein Unterordner existiert (zip entpackt in eigenen Ordner)
    subdirs = [d for d in os.listdir(WORK_DIR)
               if os.path.isdir(os.path.join(WORK_DIR, d))]
    if subdirs and not os.path.exists(os.path.join(WORK_DIR, "CMakeLists.txt")):
        WORK_DIR = os.path.join(WORK_DIR, subdirs[0])
        log(f"  Unterordner: {WORK_DIR}")

    os.chdir(WORK_DIR)
    files = os.listdir(".")
    log(f"  Arbeitsverzeichnis: {WORK_DIR}")
    log(f"  Inhalt ({len(files)} Einträge): {sorted(files)}")

In [ ]:
# ── Schritt 2: System-Abhängigkeiten installieren ──────────────────────────────
import subprocess

def run_cmd(cmd, label):
    with Timer(label):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            log(f"  STDERR: {r.stderr[-500:]}", "error")
            raise RuntimeError(f"Fehlgeschlagen: {cmd}")
        return r.stdout

run_cmd("apt-get update -qq", "apt-get update")
run_cmd("apt-get install -y -qq cmake g++ python3-dev", "cmake + g++ installieren")

cmake_ver = subprocess.run("cmake --version", shell=True, capture_output=True, text=True).stdout.split("\n")[0]
gpp_ver   = subprocess.run("g++ --version",   shell=True, capture_output=True, text=True).stdout.split("\n")[0]
log(f"  {cmake_ver}")
log(f"  {gpp_ver}")

In [ ]:
# ── Schritt 3: Python-Abhängigkeiten installieren ─────────────────────────────
with Timer("Python-Pakete installieren"):
    r = subprocess.run(
        "pip install -q stable-baselines3>=2.3 sb3-contrib gymnasium numpy tensorboard",
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        log(r.stderr[-500:], "error")
        raise RuntimeError("pip install fehlgeschlagen")

import stable_baselines3 as sb3
import torch
log(f"  stable-baselines3=={sb3.__version__}")
log(f"  torch=={torch.__version__}")

In [ ]:
# ── Schritt 4: C++-Extension kompilieren ──────────────────────────────────────
# Das ist der Schritt wo stoneforge_sim.so für Linux gebaut wird.
# Dauert ~90-120s wegen FetchContent (pybind11 + nlohmann_json von GitHub).

build_dir = os.path.join(WORK_DIR, "build")
os.makedirs(build_dir, exist_ok=True)

with Timer("CMake configure"):
    cmake_cfg = (
        f"cmake -S {WORK_DIR} -B {build_dir} "
        f"-DCMAKE_BUILD_TYPE=Release "
        f"-DBUILD_PYTHON_BINDINGS=ON "
        f"-DBUILD_HEADLESS_RUNNER=OFF "
        f"-DBUILD_RAYLIB_CLIENT=OFF "
        f"-DBUILD_SDL_CLIENT=OFF "
        f"-DPython3_EXECUTABLE={sys.executable}"
    )
    r = subprocess.run(cmake_cfg, shell=True, capture_output=True, text=True)
    # Relevante Zeilen aus CMake-Output zeigen
    for line in r.stdout.splitlines():
        if any(k in line for k in ["Found", "pybind", "Python", "Error", "Warning", "stoneforge"]):
            log(f"  cmake: {line.strip()}")
    if r.returncode != 0:
        log(r.stderr[-1000:], "error")
        raise RuntimeError("CMake configure fehlgeschlagen")

with Timer("C++ kompilieren (stoneforge_sim.so)"):
    r = subprocess.run(
        f"cmake --build {build_dir} -j {os.cpu_count()}",
        shell=True, capture_output=True, text=True
    )
    for line in r.stdout.splitlines():
        if any(k in line for k in ["stoneforge", "error", "warning", "Linking", "Built"]):
            log(f"  make: {line.strip()}")
    if r.returncode != 0:
        log(r.stderr[-1000:], "error")
        raise RuntimeError("Build fehlgeschlagen")

# .so-Datei bestätigen
so_files = glob.glob(f"{build_dir}/**/*.so", recursive=True)
for f in so_files:
    size_kb = os.path.getsize(f) / 1024
    log(f"  Binary: {f}  ({size_kb:.0f} KB)")

In [ ]:
# ── Schritt 5: Import testen + GPU-Info ───────────────────────────────────────
sys.path.insert(0, build_dir)
sys.path.insert(0, os.path.join(WORK_DIR, "python"))
sys.path.insert(0, os.path.join(WORK_DIR, "scripts"))

with Timer("Import + Env-Test"):
    import stoneforge_sim
    from stoneforge_env import StoneforgeWorldEnv

    env = StoneforgeWorldEnv(exit_min=5, exit_max=12)
    obs, _ = env.reset(seed=42)
    action = env.action_space.sample()
    obs2, r, term, trunc, info = env.step(action)
    log(f"  Obs shape: {obs.shape}  |  Action space: {env.action_space}")
    log(f"  Schritt OK: reward={r:.3f}, term={term}, info={info}")

log("")
log("═══ GPU-Info ═══")
log(f"  CUDA verfügbar : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        log(f"  GPU {i}: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB  |  "
            f"SM: {p.multi_processor_count}")
    device = "cuda"
elif torch.backends.mps.is_available():
    log("  Apple MPS verfügbar")
    device = "mps"
else:
    log("  Kein GPU — Training auf CPU", "warning")
    device = "cpu"
log(f"  → Training-Device: {device}")

In [ ]:
# ── Schritt 6: Training ───────────────────────────────────────────────────────
import numpy as np
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.utils import set_random_seed
from sb3_contrib import RecurrentPPO

EVAL_SEEDS    = list(range(7000, 7050))
MAX_EVAL_STEPS = 4000

# ── Progress + Log Callback ───────────────────────────────────────────────────
class TrainingLogger(BaseCallback):
    """Loggt FPS, ETA und Eval-Ergebnisse in Datei + Notebook."""

    def __init__(self, total_steps, log_every=10_000):
        super().__init__(verbose=0)
        self.total_steps = total_steps
        self.log_every   = log_every
        self.start_time  = None
        self.best_rate   = -1.0
        self._step_times = []

    def _on_training_start(self):
        self.start_time = time.time()
        log("")
        log("═══ Training gestartet ═══")
        log(f"  Algo       : {self.model.__class__.__name__}")
        log(f"  Device     : {self.model.device}")
        log(f"  Policy     : {self.model.policy.__class__.__name__}")
        log(f"  Timesteps  : {self.total_steps:,}")
        log(f"  n_envs     : {self.model.n_envs}")
        log("")

    def _on_step(self):
        if self.n_calls % self.log_every == 0:
            elapsed   = time.time() - self.start_time
            steps     = self.num_timesteps
            progress  = steps / self.total_steps
            fps       = steps / elapsed if elapsed > 0 else 0
            remaining = (self.total_steps - steps) / fps if fps > 0 else 0

            bar_len  = 30
            filled   = int(bar_len * progress)
            bar      = "█" * filled + "░" * (bar_len - filled)

            log(
                f"[{bar}] {progress:5.1%}  "
                f"steps={steps:>9,}  "
                f"FPS={fps:>6.0f}  "
                f"elapsed={fmt_time(elapsed)}  "
                f"ETA={fmt_time(remaining)}"
            )
        return True

    def _on_training_end(self):
        elapsed = time.time() - self.start_time
        fps     = self.total_steps / elapsed if elapsed > 0 else 0
        log("")
        log("═══ Training abgeschlossen ═══")
        log(f"  Gesamtdauer : {fmt_time(elapsed)}")
        log(f"  Ø FPS       : {fps:.0f}")
        log(f"  Beste SR    : {self.best_rate:.1%}")


# ── Eval Callback ─────────────────────────────────────────────────────────────
class SeedEvalCallback(BaseCallback):
    def __init__(self, eval_freq, save_path, n_episodes=50,
                 exit_min=35, exit_max=45, training_logger=None):
        super().__init__(verbose=0)
        self.eval_freq        = eval_freq
        self.save_path        = save_path
        self.n_episodes       = n_episodes
        self.exit_min         = exit_min
        self.exit_max         = exit_max
        self._best_rate       = -1.0
        self._training_logger = training_logger
        os.makedirs(save_path, exist_ok=True)

    def _on_step(self):
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            t_eval = time.time()
            rate   = self._run_eval()
            dur    = time.time() - t_eval

            status = "★ NEUES BESTES" if rate > self._best_rate else ""
            log(f"  ┌─ EVAL @ {self.num_timesteps:,} steps ──────────────────────")
            log(f"  │  Success Rate : {rate:.1%}  (bisher best: {self._best_rate:.1%})  {status}")
            log(f"  │  Eval-Dauer   : {dur:.1f}s")

            if rate > self._best_rate:
                self._best_rate = rate
                if self._training_logger:
                    self._training_logger.best_rate = rate
                path = os.path.join(self.save_path, "best_model")
                self.model.save(path)
                log(f"  │  → Gespeichert : {path}.zip")
            log(f"  └─────────────────────────────────────────────")

            self.logger.record("eval/success_rate", rate)
            self.logger.dump(self.num_timesteps)
        return True

    def _run_eval(self):
        env = StoneforgeWorldEnv(exit_min=self.exit_min, exit_max=self.exit_max)
        is_recurrent = isinstance(self.model, RecurrentPPO)
        successes = 0
        seeds = EVAL_SEEDS[:self.n_episodes]

        for seed in seeds:
            obs, _ = env.reset(seed=seed)
            done, steps, reached = False, 0, False
            lstm_states, ep_start = None, np.ones((1,), dtype=bool)
            while not done and steps < MAX_EVAL_STEPS:
                if is_recurrent:
                    action, lstm_states = self.model.predict(
                        obs.reshape(1, -1), state=lstm_states,
                        episode_start=ep_start, deterministic=False)
                    action = int(action[0])
                    ep_start = np.zeros((1,), dtype=bool)
                else:
                    action, _ = self.model.predict(obs, deterministic=False)
                    action = int(action)
                obs, _, term, trunc, info = env.step(action)
                steps += 1
                if info.get("reached_exit"): reached = True
                done = term or trunc
            successes += int(reached)
        return successes / len(seeds)


# ── Modell-Konfigurationen ────────────────────────────────────────────────────
RPPO_KWARGS = dict(
    policy="MlpLstmPolicy",
    n_steps=256, batch_size=8, n_epochs=10,
    learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.05, vf_coef=0.5,
    policy_kwargs=dict(n_lstm_layers=1, lstm_hidden_size=256, shared_lstm=False),
    verbose=0, tensorboard_log="/kaggle/working/logs/tensorboard/",
)
PPO_KWARGS = dict(
    policy="MlpPolicy",
    n_steps=2048, batch_size=256, n_epochs=10,
    learning_rate=3e-4, gamma=0.999, gae_lambda=0.95,
    clip_range=0.2, ent_coef=0.05, vf_coef=0.5,
    verbose=0, tensorboard_log="/kaggle/working/logs/tensorboard/",
)
DQN_KWARGS = dict(
    policy="MlpPolicy",
    learning_rate=1e-4, buffer_size=200_000, learning_starts=10_000,
    batch_size=256, gamma=0.999, exploration_fraction=0.5,
    exploration_final_eps=0.05, train_freq=4, target_update_interval=1000,
    verbose=0, tensorboard_log="/kaggle/working/logs/tensorboard/",
)
A2C_KWARGS = dict(
    policy="MlpPolicy",
    n_steps=2048, learning_rate=7e-4, gamma=0.999,
    gae_lambda=1.0, ent_coef=0.05, vf_coef=0.5,
    verbose=0, tensorboard_log="/kaggle/working/logs/tensorboard/",
)

# ── Training starten ──────────────────────────────────────────────────────────
log(f"Erstelle {N_ENVS} Environments ...")
env = make_vec_env(
    lambda: StoneforgeWorldEnv(exit_min=EXIT_MIN, exit_max=EXIT_MAX),
    n_envs=N_ENVS
)

KWARGS_MAP = {"rppo": RPPO_KWARGS, "ppo": PPO_KWARGS,
              "dqn": DQN_KWARGS,   "a2c": A2C_KWARGS}
CLS_MAP    = {"rppo": RecurrentPPO, "ppo": PPO, "dqn": DQN, "a2c": A2C}

ModelCls = CLS_MAP[ALGO]
kwargs   = KWARGS_MAP[ALGO]
model    = ModelCls(env=env, device=device, **kwargs)

progress_cb = TrainingLogger(total_steps=TIMESTEPS, log_every=LOG_EVERY)
eval_cb     = SeedEvalCallback(
    eval_freq=max(1, EVAL_FREQ // N_ENVS),
    save_path=SAVE_DIR,
    n_episodes=50, exit_min=35, exit_max=45,
    training_logger=progress_cb,
)

model.learn(
    total_timesteps=TIMESTEPS,
    callback=[progress_cb, eval_cb],
    tb_log_name=f"{ALGO}_kaggle",
    reset_num_timesteps=True,
)

# Finales Modell sichern
final_path = os.path.join(SAVE_DIR, "final_model")
model.save(final_path)
log(f"Finales Modell gespeichert: {final_path}.zip")

In [ ]:
# ── Schritt 7: Finale Evaluation + Zusammenfassung ────────────────────────────
with Timer("Finale Evaluation (50 Seeds, deterministisch)"):
    best_path = os.path.join(SAVE_DIR, "best_model.zip")
    if not os.path.exists(best_path):
        log(f"Kein best_model.zip unter {best_path}", "warning")
    else:
        eval_model = None
        for Cls in [RecurrentPPO, PPO, A2C, DQN]:
            try:
                eval_model = Cls.load(best_path, device="cpu")
                break
            except Exception:
                pass

        if eval_model is None:
            log("Modell konnte nicht geladen werden", "error")
        else:
            eval_env = StoneforgeWorldEnv(exit_min=35, exit_max=45)
            is_recurrent = isinstance(eval_model, RecurrentPPO)
            succ, lens, rets = 0, [], []

            for seed in EVAL_SEEDS:
                obs, _ = eval_env.reset(seed=seed)
                done, steps, ep_ret, reached = False, 0, 0.0, False
                lstm_states, ep_start = None, np.ones((1,), dtype=bool)
                while not done and steps < MAX_EVAL_STEPS:
                    if is_recurrent:
                        action, lstm_states = eval_model.predict(
                            obs.reshape(1, -1), state=lstm_states,
                            episode_start=ep_start, deterministic=True)
                        action = int(action[0])
                        ep_start = np.zeros((1,), dtype=bool)
                    else:
                        action, _ = eval_model.predict(obs, deterministic=True)
                        action = int(action)
                    obs, r, term, trunc, info = eval_env.step(action)
                    ep_ret += float(r); steps += 1
                    if info.get("reached_exit"): reached = True
                    done = term or trunc
                succ += int(reached); lens.append(steps); rets.append(ep_ret)

            log("")
            log("╔══════════════════════════════════════╗")
            log(f"║  ERGEBNIS — {ALGO.upper():8}  deterministisch  ║")
            log("╠══════════════════════════════════════╣")
            log(f"║  Erfolge      : {succ:>3} / 50              ║")
            log(f"║  Success Rate : {succ/50:>6.1%}               ║")
            log(f"║  Ø Ep-Länge   : {np.mean(lens):>7.1f} Steps          ║")
            log(f"║  Ø Return     : {np.mean(rets):>7.2f}               ║")
            log(f"║  Std Ep-Länge : {np.std(lens):>7.1f}               ║")
            log("╚══════════════════════════════════════╝")

# Gespeicherte Dateien auflisten
log("")
log("═══ Output-Dateien ═══")
for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d not in ["__pycache__", ".ipynb_checkpoints"]]
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        unit = "KB" if size < 1e6 else "MB"
        val  = size/1e3 if size < 1e6 else size/1e6
        log(f"  {path.replace('/kaggle/working/', '')}  ({val:.1f} {unit})")

log("")
log(f"Log-Datei: {LOG_FILE}  (Download unter Output → Files)")
log("Fertig.")